In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import NMF
from scipy.sparse.linalg import svds
import os

# Create output dirs
os.makedirs("results/svd", exist_ok=True)
os.makedirs("results/nmf", exist_ok=True)

print("--- MATRIX FACTORIZATION LAB ---")
print("Student Roll No: 24BAD067 \n")

# 1. Load the dataset
print("Loading MovieLens 100k dataset...")
cols = ['user_id', 'movie_id', 'rating', 'timestamp']
train_df = pd.read_csv('/Users/manojmj/PycharmProjects/machinelearning subject/expno10/ml-100k/u1.base', sep='\t', names=cols, encoding='latin-1')
test_df = pd.read_csv('/Users/manojmj/PycharmProjects/machinelearning subject/expno10/ml-100k/u1.test', sep='\t', names=cols, encoding='latin-1')

# Load movie names for recommendations
item_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
items_df = pd.read_csv('/Users/manojmj/PycharmProjects/machinelearning subject/expno10/ml-100k/u.item', sep='|', names=item_cols, encoding='latin-1', usecols=['movie_id', 'movie_title'])
movie_dict = dict(zip(items_df.movie_id, items_df.movie_title))

n_users = max(train_df.user_id.max(), test_df.user_id.max())
n_items = max(train_df.movie_id.max(), test_df.movie_id.max())

print(f"Dataset loaded: {n_users} users, {n_items} movies.")
print(f"Train set interactions: {len(train_df)}")
print(f"Test set interactions: {len(test_df)}\n")

# 2. Data Preprocessing: Create User-Item Interaction Matrix
R_df = train_df.pivot(index='user_id', columns='movie_id', values='rating')
# Fill missing movies to ensure matrix dimensions match max users/movies
R_df = R_df.reindex(index=range(1, n_users + 1), columns=range(1, n_items + 1))
R = R_df.fillna(0).values

print("--- SCENARIO 1: SVD ---")
# Normalizing array by mean centering
user_ratings_mean = np.mean(R, axis=1)
R_demeaned = R - user_ratings_mean.reshape(-1, 1)

# Analysis: Effect of number of latent factors (k)
k_values = [5, 10, 20, 50, 100]
rmse_list_svd = []
mae_list_svd = []

best_k = 50
R_pred_best_svd = None

print("Evaluating SVD for different k values...")
for k in k_values:
    # Apply SVD
    U, sigma, Vt = svds(R_demeaned, k=k)
    sigma = np.diag(sigma)

    # Reconstruct the matrix
    all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.reshape(-1, 1)

    if k == best_k:
        R_pred_best_svd = all_user_predicted_ratings

    # Predict and evaluate on test set
    test_preds = []
    test_actuals = []
    for _, row in test_df.iterrows():
        u = int(row['user_id']) - 1
        i = int(row['movie_id']) - 1
        test_actuals.append(row['rating'])
        # Clip predictions to 1-5
        pred = max(1, min(5, all_user_predicted_ratings[u, i]))
        test_preds.append(pred)

    rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))
    mae = mean_absolute_error(test_actuals, test_preds)
    rmse_list_svd.append(rmse)
    mae_list_svd.append(mae)
    print(f"  k={k:3d} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}")

# Plot Error vs k
plt.figure()
plt.plot(k_values, rmse_list_svd, marker='o', label='RMSE')
plt.plot(k_values, mae_list_svd, marker='x', label='MAE')
plt.title('SVD: Error metrics vs Number of Latent Factors (k)')
plt.xlabel('Number of Latent Factors (k)')
plt.ylabel('Error')
plt.legend()
plt.savefig("results/svd/error_vs_k.png")
plt.close()

# Heatmap of Original vs Reconstructed
sample_u, sample_i = 50, 50
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.heatmap(R[:sample_u, :sample_i], cmap="viridis", vmin=0, vmax=5)
plt.title("Original Rating Matrix (Sample)")
plt.xlabel("Movie ID")
plt.ylabel("User ID")

plt.subplot(1, 2, 2)
sns.heatmap(R_pred_best_svd[:sample_u, :sample_i], cmap="viridis", vmin=0, vmax=5)
plt.title(f"Reconstructed Matrix (k={best_k})")
plt.xlabel("Movie ID")
plt.savefig("results/svd/heatmap_comparison.png")
plt.close()

# Generate Top-N recommendations for a sample user
target_user = 1
u_idx = target_user - 1
user_seen_movies = train_df[train_df['user_id'] == target_user]['movie_id'].values
preds = R_pred_best_svd[u_idx, :]
# Sort descending
top_indices = preds.argsort()[::-1]

recommendations_svd = []
count = 0
print(f"\nTop 10 SVD Recommendations for User {target_user}:")
for idx in top_indices:
    movie_id = idx + 1
    if movie_id not in user_seen_movies:
        title = movie_dict.get(movie_id, "Unknown Movie")
        score = preds[idx]
        print(f"  {count+1}. {title} (Predicted Rating: {score:.2f})")
        recommendations_svd.append((title, score))
        count += 1
        if count == 10:
            break


plt.figure(figsize=(10, 6))
recom_titles = [r[0] for r in recommendations_svd]
recom_scores = [r[1] for r in recommendations_svd]
sns.barplot(x=recom_scores, y=recom_titles, hue=recom_titles, palette="Blues_r", legend=False)
plt.title(f"SVD: Top 10 Recommended Movies for User {target_user}")
plt.xlabel("Predicted Rating")
plt.xlim(3, 5)
plt.savefig("results/svd/top_recommendations.png", bbox_inches='tight')
plt.close()




--- MATRIX FACTORIZATION LAB ---
Student Roll No: 24BAD067 

Loading MovieLens 100k dataset...
Dataset loaded: 943 users, 1682 movies.
Train set interactions: 80000
Test set interactions: 20000

--- SCENARIO 1: SVD ---
Evaluating SVD for different k values...
  k=  5 -> RMSE: 2.5882, MAE: 2.3302
  k= 10 -> RMSE: 2.5543, MAE: 2.2975
  k= 20 -> RMSE: 2.5494, MAE: 2.2912
  k= 50 -> RMSE: 2.6200, MAE: 2.3639
  k=100 -> RMSE: 2.7063, MAE: 2.4544

Top 10 SVD Recommendations for User 1:
  1. Raiders of the Lost Ark (1981) (Predicted Rating: 3.38)
  2. Secrets & Lies (1996) (Predicted Rating: 3.02)
  3. Alien (1979) (Predicted Rating: 2.82)
  4. Strictly Ballroom (1992) (Predicted Rating: 2.21)
  5. Fargo (1996) (Predicted Rating: 2.09)
  6. Apocalypse Now (1979) (Predicted Rating: 2.08)
  7. Silence of the Lambs, The (1991) (Predicted Rating: 2.07)
  8. Usual Suspects, The (1995) (Predicted Rating: 2.07)
  9. Leaving Las Vegas (1995) (Predicted Rating: 2.04)
  10. Close Shave, A (1995) (Predi